In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os

PROJECT = "/content/drive/MyDrive/race_rc_project"

PROCESSED_DIR = f"{PROJECT}/data/processed"
MODEL_B_DIR = f"{PROJECT}/models/model_b"
RESULTS_DIR = f"{PROJECT}/results"
MODEL_B_RESULTS_DIR = f"{RESULTS_DIR}/model_b_full"

os.makedirs(MODEL_B_DIR, exist_ok=True)
os.makedirs(MODEL_B_RESULTS_DIR, exist_ok=True)

TRAIN_PATH = f"{PROCESSED_DIR}/model_b_train_full.csv"
VAL_PATH = f"{PROCESSED_DIR}/model_b_val_full.csv"
TEST_PATH = f"{PROCESSED_DIR}/model_b_test_full.csv"

print("PROJECT exists:", os.path.exists(PROJECT))
print("PROCESSED_DIR exists:", os.path.exists(PROCESSED_DIR))
print("MODEL_B_DIR exists:", os.path.exists(MODEL_B_DIR))
print("MODEL_B_RESULTS_DIR exists:", os.path.exists(MODEL_B_RESULTS_DIR))

for path in [TRAIN_PATH, VAL_PATH, TEST_PATH]:
    print(path, "->", os.path.exists(path))

Mounted at /content/drive
PROJECT exists: True
PROCESSED_DIR exists: True
MODEL_B_DIR exists: True
MODEL_B_RESULTS_DIR exists: True
/content/drive/MyDrive/race_rc_project/data/processed/model_b_train_full.csv -> True
/content/drive/MyDrive/race_rc_project/data/processed/model_b_val_full.csv -> True
/content/drive/MyDrive/race_rc_project/data/processed/model_b_test_full.csv -> True


In [2]:
!pip -q install rouge-score nltk

  Preparing metadata (setup.py) ... done


In [3]:
import re
import ast
import json
import math
import string
import random
import warnings
from collections import Counter

import numpy as np
import pandas as pd
import joblib

import nltk
nltk.download("punkt", quiet=True)
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)

from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score
from rouge_score import rouge_scorer

from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

FULL_MODE = True

# Full experiment settings
MAX_NEGATIVE_CANDIDATES_PER_QUESTION = 40
FEATURE_CHUNK_SIZE = 100000
SCORE_CHUNK_SIZE = 100000

print("FULL_MODE:", FULL_MODE)
print("MAX_NEGATIVE_CANDIDATES_PER_QUESTION:", MAX_NEGATIVE_CANDIDATES_PER_QUESTION)

FULL_MODE: True
MAX_NEGATIVE_CANDIDATES_PER_QUESTION: 40


In [4]:
train_df = pd.read_csv(TRAIN_PATH)
val_df = pd.read_csv(VAL_PATH)
test_df = pd.read_csv(TEST_PATH)

print("Train shape:", train_df.shape)
print("Val shape:", val_df.shape)
print("Test shape:", test_df.shape)

print("\nColumns:")
print(train_df.columns.tolist())

display(train_df.head(3))

Train shape: (70280, 16)
Val shape: (8785, 16)
Test shape: (8786, 16)

Columns:
['question_uid', 'id', 'article', 'article_nopunct', 'question', 'question_nopunct', 'correct_answer', 'correct_answer_nopunct', 'gold_answer_label', 'distractor_1', 'distractor_2', 'distractor_3', 'distractor_label_1', 'distractor_label_2', 'distractor_label_3', 'article_sentences']


,question_uid,id,article,article_nopunct,question,question_nopunct,correct_answer,correct_answer_nopunct,gold_answer_label,distractor_1,distractor_2,distractor_3,distractor_label_1,distractor_label_2,distractor_label_3,article_sentences
0,q_2112,middle3221.txt,everyone knows that chickens lay eggs . most p...,everyone knows that chickens lay eggs most peo...,egg shells can be used to _ .,egg shells can be used to,protect the young animals,protect the young animals,D,make eggs more beautiful,make eggs more tasty,let the young animals dry,A,B,C,"['everyone knows that chickens lay eggs .', 'm..."
1,q_46988,high17034.txt,"more surprising,perhaps,than the present diffi...",more surprisingperhapsthan the present difficu...,which of the following can be presented as the...,which of the following can be presented as the...,many types of family arrangements have become ...,many types of family arrangements have become ...,A,a typical american family is made up of only a...,americans prefer to have more kids than before.,there are no nuclear families any more.,B,C,D,"['more surprising,perhaps,than the present dif..."
2,q_3453,middle3657.txt,"on a cold winter day, a fox told mother bear t...",on a cold winter day a fox told mother bear th...,the fox asked mother bear _ .,the fox asked mother bear,"if she pulled the tail out of the water, she w...",if she pulled the tail out of the water she wo...,D,to jump into the water,to sit by the lake for a long time,to put her tail down into the water and never ...,A,B,C,"['on a cold winter day, a fox told mother bear..."


In [5]:
required_cols = [
    "question_uid",
    "id",
    "article",
    "article_nopunct",
    "question",
    "question_nopunct",
    "correct_answer",
    "correct_answer_nopunct",
    "gold_answer_label",
    "distractor_1",
    "distractor_2",
    "distractor_3",
    "article_sentences"
]

def validate_model_b_df(df, name):
    print(f"\nValidating {name}...")

    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"{name} missing columns: {missing}")

    print("All required columns present.")
    print("Rows:", len(df))
    print("Unique question_uid:", df["question_uid"].nunique())
    print("Duplicate question_uid:", df["question_uid"].duplicated().sum())

    nulls = df[required_cols].isnull().sum()
    print("\nNull counts:")
    print(nulls[nulls > 0])

validate_model_b_df(train_df, "train_df")
validate_model_b_df(val_df, "val_df")
validate_model_b_df(test_df, "test_df")


Validating train_df...
All required columns present.
Rows: 70280
Unique question_uid: 70280
Duplicate question_uid: 0

Null counts:
question_nopunct          16
correct_answer_nopunct    15
dtype: int64

Validating val_df...
All required columns present.
Rows: 8785
Unique question_uid: 8785
Duplicate question_uid: 0

Null counts:
question_nopunct          2
correct_answer_nopunct    2
dtype: int64

Validating test_df...
All required columns present.
Rows: 8786
Unique question_uid: 8786
Duplicate question_uid: 0

Null counts:
question_nopunct          3
correct_answer_nopunct    4
dtype: int64


In [6]:
tfidf_path = f"{MODEL_B_DIR}/model_b_tfidf_vectorizer_full.pkl"

if os.path.exists(tfidf_path):
    model_b_tfidf = joblib.load(tfidf_path)
    print("Loaded existing TF-IDF vectorizer:", tfidf_path)
else:
    train_corpus = (
        train_df["article"].fillna("") + " " +
        train_df["question"].fillna("") + " " +
        train_df["correct_answer"].fillna("")
    ).tolist()

    model_b_tfidf = TfidfVectorizer(
        lowercase=True,
        stop_words="english",
        max_features=50000,
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.95
    )

    model_b_tfidf.fit(train_corpus)
    joblib.dump(model_b_tfidf, tfidf_path)

print("TF-IDF vocabulary size:", len(model_b_tfidf.vocabulary_))
print("Vectorizer path:", tfidf_path)

Loaded existing TF-IDF vectorizer: /content/drive/MyDrive/race_rc_project/models/model_b/model_b_tfidf_vectorizer_full.pkl
TF-IDF vocabulary size: 50000
Vectorizer path: /content/drive/MyDrive/race_rc_project/models/model_b/model_b_tfidf_vectorizer_full.pkl


In [7]:
STOPWORDS = set(ENGLISH_STOP_WORDS)

def safe_text(x):
    if pd.isna(x):
        return ""
    return str(x)

def normalize_text(text):
    text = safe_text(text).lower()
    text = re.sub(r"\s+", " ", text).strip()
    return text

def remove_punctuation(text):
    return safe_text(text).translate(str.maketrans("", "", string.punctuation))

def tokenize_words(text):
    text = normalize_text(remove_punctuation(text))
    return re.findall(r"\b[a-zA-Z][a-zA-Z0-9]*\b", text)

def content_words(text):
    return [t for t in tokenize_words(text) if t not in STOPWORDS and len(t) > 2]

def word_overlap(a, b):
    a_words = set(content_words(a))
    b_words = set(content_words(b))
    if not a_words or not b_words:
        return 0.0
    return len(a_words & b_words) / len(a_words | b_words)

def char_overlap(a, b):
    a_chars = set(normalize_text(remove_punctuation(a)).replace(" ", ""))
    b_chars = set(normalize_text(remove_punctuation(b)).replace(" ", ""))
    if not a_chars or not b_chars:
        return 0.0
    return len(a_chars & b_chars) / len(a_chars | b_chars)

def candidate_frequency_in_article(candidate, article):
    cand = normalize_text(candidate)
    art = normalize_text(article)
    if not cand:
        return 0
    return art.count(cand)

def parse_sentence_list(x):
    if isinstance(x, list):
        return x

    x = safe_text(x)

    try:
        parsed = ast.literal_eval(x)
        if isinstance(parsed, list):
            return [safe_text(s) for s in parsed if safe_text(s).strip()]
    except:
        pass

    parts = re.split(r"(?<=[.!?])\s+", x)
    return [p.strip() for p in parts if p.strip()]

print("Text helpers ready.")

Text helpers ready.


In [8]:
def generate_ngrams(tokens, n):
    return [" ".join(tokens[i:i+n]) for i in range(len(tokens)-n+1)]

def extract_candidate_phrases(article, question=None, correct_answer=None, max_candidates=40):
    article_norm = normalize_text(article)
    question_norm = normalize_text(question)
    answer_norm = normalize_text(correct_answer)

    tokens = content_words(article_norm)

    unigrams = tokens
    bigrams = generate_ngrams(tokens, 2)
    trigrams = generate_ngrams(tokens, 3)

    all_candidates = unigrams + bigrams + trigrams
    counts = Counter(all_candidates)

    cleaned = []
    seen = set()

    for cand, freq in counts.most_common():
        cand_norm = normalize_text(cand)

        if cand_norm in seen:
            continue
        if len(cand_norm) < 3:
            continue
        if cand_norm == answer_norm:
            continue
        if answer_norm and cand_norm in answer_norm:
            continue
        if answer_norm and answer_norm in cand_norm:
            continue
        if answer_norm and word_overlap(cand_norm, answer_norm) >= 0.85:
            continue
        if question_norm and cand_norm == question_norm:
            continue

        cleaned.append(cand_norm)
        seen.add(cand_norm)

        if len(cleaned) >= max_candidates:
            break

    return cleaned

def get_gold_distractors(row):
    distractors = [
        safe_text(row["distractor_1"]).strip(),
        safe_text(row["distractor_2"]).strip(),
        safe_text(row["distractor_3"]).strip()
    ]
    return [d for d in distractors if d]

def build_candidate_rows_for_question(row, max_negative_candidates=40):
    question_uid = row["question_uid"]
    article = safe_text(row["article"])
    question = safe_text(row["question"])
    correct_answer = safe_text(row["correct_answer"])

    gold_distractors = get_gold_distractors(row)
    gold_norm = set(normalize_text(d) for d in gold_distractors)

    rows = []

    # Positive candidates: original RACE wrong answers
    for d in gold_distractors:
        rows.append({
            "question_uid": question_uid,
            "article": article,
            "question": question,
            "correct_answer": correct_answer,
            "candidate": d,
            "label": 1,
            "source": "gold_distractor"
        })

    # Negative candidates: extracted from passage
    extracted = extract_candidate_phrases(
        article=article,
        question=question,
        correct_answer=correct_answer,
        max_candidates=max_negative_candidates
    )

    for cand in extracted:
        cand_norm = normalize_text(cand)
        if cand_norm in gold_norm:
            continue

        rows.append({
            "question_uid": question_uid,
            "article": article,
            "question": question,
            "correct_answer": correct_answer,
            "candidate": cand,
            "label": 0,
            "source": "extracted_negative"
        })

    return rows

sample = train_df.iloc[0]
sample_rows = build_candidate_rows_for_question(sample, MAX_NEGATIVE_CANDIDATES_PER_QUESTION)
sample_candidate_df = pd.DataFrame(sample_rows)

print("Sample question:", sample["question"])
print("Correct answer:", sample["correct_answer"])
print("Gold distractors:", get_gold_distractors(sample))
print("Candidate rows:", sample_candidate_df.shape)
display(sample_candidate_df.head(10))
print(sample_candidate_df["label"].value_counts())

Sample question: egg shells can be used to _ .
Correct answer: protect the young animals
Gold distractors: ['make eggs more beautiful', 'make eggs more tasty', 'let the young animals dry']
Candidate rows: (43, 7)


,question_uid,article,question,correct_answer,candidate,label,source
0,q_2112,everyone knows that chickens lay eggs . most p...,egg shells can be used to _ .,protect the young animals,make eggs more beautiful,1,gold_distractor
1,q_2112,everyone knows that chickens lay eggs . most p...,egg shells can be used to _ .,protect the young animals,make eggs more tasty,1,gold_distractor
2,q_2112,everyone knows that chickens lay eggs . most p...,egg shells can be used to _ .,protect the young animals,let the young animals dry,1,gold_distractor
3,q_2112,everyone knows that chickens lay eggs . most p...,egg shells can be used to _ .,protect the young animals,eggs,0,extracted_negative
4,q_2112,everyone knows that chickens lay eggs . most p...,egg shells can be used to _ .,protect the young animals,lay,0,extracted_negative
5,q_2112,everyone knows that chickens lay eggs . most p...,egg shells can be used to _ .,protect the young animals,lay eggs,0,extracted_negative
6,q_2112,everyone knows that chickens lay eggs . most p...,egg shells can be used to _ .,protect the young animals,babies,0,extracted_negative
7,q_2112,everyone knows that chickens lay eggs . most p...,egg shells can be used to _ .,protect the young animals,animals lay,0,extracted_negative
8,q_2112,everyone knows that chickens lay eggs . most p...,egg shells can be used to _ .,protect the young animals,animals lay eggs,0,extracted_negative
9,q_2112,everyone knows that chickens lay eggs . most p...,egg shells can be used to _ .,protect the young animals,mother,0,extracted_negative


label
0    40
1     3
Name: count, dtype: int64


In [9]:
def build_candidate_dataset_full(df, name, output_path, max_negative_candidates=40):
    if os.path.exists(output_path):
        print(f"{name}: candidate file already exists. Loading:", output_path)
        loaded = pd.read_csv(output_path)
        print(f"{name} shape:", loaded.shape)
        print(loaded["label"].value_counts())
        return loaded

    all_rows = []

    for i, row in df.iterrows():
        rows = build_candidate_rows_for_question(
            row,
            max_negative_candidates=max_negative_candidates
        )
        all_rows.extend(rows)

        if (i + 1) % 5000 == 0:
            print(f"{name}: processed {i + 1}/{len(df)} questions")

    candidate_df = pd.DataFrame(all_rows)
    candidate_df.to_csv(output_path, index=False)

    print(f"\n{name} candidate dataset saved:", output_path)
    print(f"{name} shape:", candidate_df.shape)
    print(f"{name} label distribution:")
    print(candidate_df["label"].value_counts())
    print(f"{name} positive rate:", candidate_df["label"].mean())

    return candidate_df

train_candidates_path = f"{MODEL_B_RESULTS_DIR}/model_b_train_candidates_full.csv"
val_candidates_path = f"{MODEL_B_RESULTS_DIR}/model_b_val_candidates_full.csv"
test_candidates_path = f"{MODEL_B_RESULTS_DIR}/model_b_test_candidates_full.csv"

train_candidates = build_candidate_dataset_full(
    train_df,
    "train",
    train_candidates_path,
    MAX_NEGATIVE_CANDIDATES_PER_QUESTION
)

val_candidates = build_candidate_dataset_full(
    val_df,
    "val",
    val_candidates_path,
    MAX_NEGATIVE_CANDIDATES_PER_QUESTION
)

test_candidates = build_candidate_dataset_full(
    test_df,
    "test",
    test_candidates_path,
    MAX_NEGATIVE_CANDIDATES_PER_QUESTION
)

train: candidate file already exists. Loading: /content/drive/MyDrive/race_rc_project/results/model_b_full/model_b_train_candidates_full.csv
train shape: (1288390, 7)
label
0    1198390
1      90000
Name: count, dtype: int64
val: candidate file already exists. Loading: /content/drive/MyDrive/race_rc_project/results/model_b_full/model_b_val_candidates_full.csv
val shape: (214697, 7)
label
0    199697
1     15000
Name: count, dtype: int64
test: candidate file already exists. Loading: /content/drive/MyDrive/race_rc_project/results/model_b_full/model_b_test_candidates_full.csv
test shape: (214740, 7)
label
0    199740
1     15000
Name: count, dtype: int64


In [10]:
def paired_cosine_sparse(A, B):
    # A and B are sparse matrices with same number of rows.
    numer = A.multiply(B).sum(axis=1).A1
    A_norm = np.sqrt(A.multiply(A).sum(axis=1).A1)
    B_norm = np.sqrt(B.multiply(B).sum(axis=1).A1)
    denom = A_norm * B_norm
    out = np.zeros_like(numer, dtype=float)
    mask = denom != 0
    out[mask] = numer[mask] / denom[mask]
    return out

def extract_features_chunk(chunk_df):
    chunk_df = chunk_df.copy()

    candidates = chunk_df["candidate"].fillna("").astype(str).tolist()
    answers = chunk_df["correct_answer"].fillna("").astype(str).tolist()
    questions = chunk_df["question"].fillna("").astype(str).tolist()
    articles = chunk_df["article"].fillna("").astype(str).tolist()

    cand_vec = model_b_tfidf.transform(candidates)
    ans_vec = model_b_tfidf.transform(answers)
    ques_vec = model_b_tfidf.transform(questions)
    art_vec = model_b_tfidf.transform(articles)

    cos_candidate_answer = paired_cosine_sparse(cand_vec, ans_vec)
    cos_candidate_question = paired_cosine_sparse(cand_vec, ques_vec)
    cos_candidate_article = paired_cosine_sparse(cand_vec, art_vec)

    rows = []

    for idx, row in chunk_df.reset_index(drop=True).iterrows():
        article = safe_text(row["article"])
        question = safe_text(row["question"])
        correct_answer = safe_text(row["correct_answer"])
        candidate = safe_text(row["candidate"])

        cand_words = content_words(candidate)
        ans_words = content_words(correct_answer)

        rows.append({
            "question_uid": row["question_uid"],
            "candidate": candidate,
            "label": int(row["label"]),

            "cos_candidate_answer": float(cos_candidate_answer[idx]),
            "cos_candidate_question": float(cos_candidate_question[idx]),
            "cos_candidate_article": float(cos_candidate_article[idx]),

            "candidate_word_len": len(cand_words),
            "answer_word_len": len(ans_words),
            "length_diff": abs(len(cand_words) - len(ans_words)),

            "char_overlap_answer": char_overlap(candidate, correct_answer),
            "word_overlap_answer": word_overlap(candidate, correct_answer),
            "word_overlap_question": word_overlap(candidate, question),

            "candidate_frequency_article": candidate_frequency_in_article(candidate, article),
            "candidate_in_question": int(normalize_text(candidate) in normalize_text(question)),
            "candidate_same_as_answer": int(normalize_text(candidate) == normalize_text(correct_answer)),
            "candidate_contains_answer": int(normalize_text(correct_answer) in normalize_text(candidate)),
            "answer_contains_candidate": int(normalize_text(candidate) in normalize_text(correct_answer))
        })

    return pd.DataFrame(rows)

def extract_features_full(candidate_path, output_path, name, chunk_size=100000):
    if os.path.exists(output_path):
        print(f"{name}: feature file already exists. Loading:", output_path)
        feature_df = pd.read_csv(output_path)
        print(f"{name} feature shape:", feature_df.shape)
        print(feature_df["label"].value_counts())
        return feature_df

    total_rows = sum(1 for _ in open(candidate_path, "r", encoding="utf-8")) - 1
    print(f"{name}: total candidate rows:", total_rows)

    first_write = True
    processed = 0

    for chunk in pd.read_csv(candidate_path, chunksize=chunk_size):
        feature_chunk = extract_features_chunk(chunk)

        feature_chunk.to_csv(
            output_path,
            mode="w" if first_write else "a",
            index=False,
            header=first_write
        )

        first_write = False
        processed += len(chunk)
        print(f"{name}: processed {processed}/{total_rows} feature rows")

    feature_df = pd.read_csv(output_path)

    print(f"\n{name} features saved:", output_path)
    print(f"{name} feature shape:", feature_df.shape)
    print(f"{name} label distribution:")
    print(feature_df["label"].value_counts())

    return feature_df

train_features_path = f"{MODEL_B_RESULTS_DIR}/model_b_train_features_full.csv"
val_features_path = f"{MODEL_B_RESULTS_DIR}/model_b_val_features_full.csv"
test_features_path = f"{MODEL_B_RESULTS_DIR}/model_b_test_features_full.csv"

train_features = extract_features_full(
    train_candidates_path,
    train_features_path,
    "train",
    FEATURE_CHUNK_SIZE
)

val_features = extract_features_full(
    val_candidates_path,
    val_features_path,
    "val",
    FEATURE_CHUNK_SIZE
)

test_features = extract_features_full(
    test_candidates_path,
    test_features_path,
    "test",
    FEATURE_CHUNK_SIZE
)

train: total candidate rows: 1288390
train: processed 100000/1288390 feature rows
train: processed 200000/1288390 feature rows
train: processed 300000/1288390 feature rows
train: processed 400000/1288390 feature rows
train: processed 500000/1288390 feature rows
train: processed 600000/1288390 feature rows
train: processed 700000/1288390 feature rows
train: processed 800000/1288390 feature rows
train: processed 900000/1288390 feature rows
train: processed 1000000/1288390 feature rows
train: processed 1100000/1288390 feature rows
train: processed 1200000/1288390 feature rows
train: processed 1288390/1288390 feature rows

train features saved: /content/drive/MyDrive/race_rc_project/results/model_b_full/model_b_train_features_full.csv
train feature shape: (1288390, 17)
train label distribution:
label
0    1198390
1      90000
Name: count, dtype: int64
val: total candidate rows: 214697
val: processed 100000/214697 feature rows
val: processed 200000/214697 feature rows
val: processed 214697/

In [11]:
feature_cols = [
    "cos_candidate_answer",
    "cos_candidate_question",
    "cos_candidate_article",
    "candidate_word_len",
    "answer_word_len",
    "length_diff",
    "char_overlap_answer",
    "word_overlap_answer",
    "word_overlap_question",
    "candidate_frequency_article",
    "candidate_in_question",
    "candidate_same_as_answer",
    "candidate_contains_answer",
    "answer_contains_candidate"
]

X_train = train_features[feature_cols]
y_train = train_features["label"].astype(int)

X_val = val_features[feature_cols]
y_val = val_features["label"].astype(int)

X_test = test_features[feature_cols]
y_test = test_features["label"].astype(int)

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)

print("\nTrain labels:")
print(y_train.value_counts())

print("\nVal labels:")
print(y_val.value_counts())

print("\nTest labels:")
print(y_test.value_counts())

X_train: (1288390, 14)
X_val: (214697, 14)
X_test: (214740, 14)

Train labels:
label
0    1198390
1      90000
Name: count, dtype: int64

Val labels:
label
0    199697
1     15000
Name: count, dtype: int64

Test labels:
label
0    199740
1     15000
Name: count, dtype: int64


In [12]:
logreg_ranker_path = f"{MODEL_B_DIR}/logistic_regression_distractor_ranker_full.pkl"

if os.path.exists(logreg_ranker_path):
    logreg_ranker = joblib.load(logreg_ranker_path)
    print("Loaded Logistic Regression ranker:", logreg_ranker_path)
else:
    logreg_ranker = Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=RANDOM_STATE,
            n_jobs=-1
        ))
    ])

    logreg_ranker.fit(X_train, y_train)
    joblib.dump(logreg_ranker, logreg_ranker_path)
    print("Saved Logistic Regression ranker:", logreg_ranker_path)

print("Logistic Regression distractor ranker ready.")

Saved Logistic Regression ranker: /content/drive/MyDrive/race_rc_project/models/model_b/logistic_regression_distractor_ranker_full.pkl
Logistic Regression distractor ranker ready.


In [13]:
rf_ranker_path = f"{MODEL_B_DIR}/random_forest_distractor_ranker_full.pkl"

if os.path.exists(rf_ranker_path):
    rf_ranker = joblib.load(rf_ranker_path)
    print("Loaded Random Forest ranker:", rf_ranker_path)
else:
    rf_ranker = RandomForestClassifier(
        n_estimators=200,
        max_depth=18,
        min_samples_leaf=3,
        class_weight="balanced_subsample",
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbose=1
    )

    rf_ranker.fit(X_train, y_train)
    joblib.dump(rf_ranker, rf_ranker_path)
    print("Saved Random Forest ranker:", rf_ranker_path)

print("Random Forest distractor ranker ready.")

[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 2 concurrent workers.
[Parallel(n_jobs=-1)]: Done  46 tasks      | elapsed:  1.3min
[Parallel(n_jobs=-1)]: Done 196 tasks      | elapsed:  5.7min
[Parallel(n_jobs=-1)]: Done 200 out of 200 | elapsed:  5.8min finished


Saved Random Forest ranker: /content/drive/MyDrive/race_rc_project/models/model_b/random_forest_distractor_ranker_full.pkl
Random Forest distractor ranker ready.


In [15]:
def score_feature_file(model, feature_path, candidate_path, output_path, model_name, chunk_size=100000):
    if os.path.exists(output_path):
        print(f"{model_name}: scored file already exists. Loading:", output_path)
        scored_df = pd.read_csv(output_path)
        print(scored_df.shape)
        return scored_df

    first_write = True
    processed = 0

    feature_chunks = pd.read_csv(feature_path, chunksize=chunk_size)
    candidate_chunks = pd.read_csv(candidate_path, chunksize=chunk_size)

    for f_chunk, c_chunk in zip(feature_chunks, candidate_chunks):
        X_chunk = f_chunk[feature_cols]

        if hasattr(model, "predict_proba"):
            scores = model.predict_proba(X_chunk)[:, 1]
        else:
            scores = model.decision_function(X_chunk)

        scored_chunk = c_chunk[[
            "question_uid",
            "question",
            "correct_answer",
            "candidate",
            "label",
            "source"
        ]].copy()

        scored_chunk["score"] = scores
        scored_chunk["model"] = model_name

        scored_chunk.to_csv(
            output_path,
            mode="w" if first_write else "a",
            index=False,
            header=first_write
        )

        first_write = False
        processed += len(scored_chunk)
        print(f"{model_name}: scored {processed} rows")

    scored_df = pd.read_csv(output_path)
    print(f"{model_name}: saved scored candidates:", output_path)
    print(scored_df.shape)
    return scored_df

lr_val_scored_path = f"{MODEL_B_RESULTS_DIR}/val_scored_logreg_full.csv"
rf_val_scored_path = f"{MODEL_B_RESULTS_DIR}/val_scored_rf_full.csv"

val_scored_lr = score_feature_file(
    logreg_ranker,
    val_features_path,
    val_candidates_path,
    lr_val_scored_path,
    "logistic_regression",
    SCORE_CHUNK_SIZE
)

val_scored_rf = score_feature_file(
    rf_ranker,
    val_features_path,
    val_candidates_path,
    rf_val_scored_path,
    "random_forest",
    SCORE_CHUNK_SIZE
)

logistic_regression: scored 100000 rows
logistic_regression: scored 200000 rows
logistic_regression: scored 214697 rows
logistic_regression: saved scored candidates: /content/drive/MyDrive/race_rc_project/results/model_b_full/val_scored_logreg_full.csv
(214697, 8)


[Parallel(n_jobs=2)]: Using backend ThreadingBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done  46 tasks      | elapsed:    0.3s
[Parallel(n_jobs=2)]: Done 196 tasks      | elapsed:    1.2s
[Parallel(n_jobs=2)]: Done 200 out of 200 | elapsed:    1.2s finished


random_forest: scored 100000 rows


[Parallel(n_jobs=2)]: Using backend ThreadingBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done  46 tasks      | elapsed:    0.3s
[Parallel(n_jobs=2)]: Done 196 tasks      | elapsed:    1.2s
[Parallel(n_jobs=2)]: Done 200 out of 200 | elapsed:    1.2s finished


random_forest: scored 200000 rows


[Parallel(n_jobs=2)]: Using backend ThreadingBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done  46 tasks      | elapsed:    0.1s
[Parallel(n_jobs=2)]: Done 196 tasks      | elapsed:    0.2s
[Parallel(n_jobs=2)]: Done 200 out of 200 | elapsed:    0.2s finished


random_forest: scored 214697 rows
random_forest: saved scored candidates: /content/drive/MyDrive/race_rc_project/results/model_b_full/val_scored_rf_full.csv
(214697, 8)


In [16]:
def candidate_similarity(a, b):
    return word_overlap(a, b)

def select_top3_for_group(group, diversity_lambda=0.25):
    group = group.copy()

    # Never choose the correct answer.
    answer = safe_text(group["correct_answer"].iloc[0])
    group = group[
        group["candidate"].apply(lambda x: normalize_text(x) != normalize_text(answer))
    ]

    group = group.sort_values("score", ascending=False)

    selected = []

    for _, row in group.iterrows():
        cand = safe_text(row["candidate"])

        if not cand.strip():
            continue

        # Avoid duplicates.
        if normalize_text(cand) in [normalize_text(x["candidate"]) for x in selected]:
            continue

        # Diversity penalty against already selected distractors.
        too_similar = False
        for chosen in selected:
            sim = candidate_similarity(cand, chosen["candidate"])
            if sim >= diversity_lambda:
                too_similar = True
                break

        if too_similar:
            continue

        selected.append(row.to_dict())

        if len(selected) == 3:
            break

    # Fallback if diversity is too strict.
    if len(selected) < 3:
        for _, row in group.iterrows():
            cand = safe_text(row["candidate"])
            if normalize_text(cand) in [normalize_text(x["candidate"]) for x in selected]:
                continue
            selected.append(row.to_dict())
            if len(selected) == 3:
                break

    return selected

def select_top3_distractors(scored_df, output_path, model_name, diversity_lambda=0.25):
    if os.path.exists(output_path):
        selected_df = pd.read_csv(output_path)
        print("Loaded selected distractors:", output_path)
        print(selected_df.shape)
        return selected_df

    rows = []

    for qid, group in scored_df.groupby("question_uid"):
        selected = select_top3_for_group(group, diversity_lambda=diversity_lambda)

        base = group.iloc[0]
        out = {
            "question_uid": qid,
            "question": base["question"],
            "correct_answer": base["correct_answer"],
            "model": model_name
        }

        for i in range(3):
            if i < len(selected):
                out[f"generated_distractor_{i+1}"] = selected[i]["candidate"]
                out[f"generated_score_{i+1}"] = selected[i]["score"]
            else:
                out[f"generated_distractor_{i+1}"] = ""
                out[f"generated_score_{i+1}"] = np.nan

        rows.append(out)

    selected_df = pd.DataFrame(rows)
    selected_df.to_csv(output_path, index=False)

    print("Saved selected distractors:", output_path)
    print(selected_df.shape)

    return selected_df

lr_val_selected_path = f"{MODEL_B_RESULTS_DIR}/val_top3_logreg_full.csv"
rf_val_selected_path = f"{MODEL_B_RESULTS_DIR}/val_top3_rf_full.csv"

val_top3_lr = select_top3_distractors(
    val_scored_lr,
    lr_val_selected_path,
    "logistic_regression",
    diversity_lambda=0.25
)

val_top3_rf = select_top3_distractors(
    val_scored_rf,
    rf_val_selected_path,
    "random_forest",
    diversity_lambda=0.25
)

display(val_top3_lr.head())
display(val_top3_rf.head())

Saved selected distractors: /content/drive/MyDrive/race_rc_project/results/model_b_full/val_top3_logreg_full.csv
(5000, 10)
Saved selected distractors: /content/drive/MyDrive/race_rc_project/results/model_b_full/val_top3_rf_full.csv
(5000, 10)


,question_uid,question,correct_answer,model,generated_distractor_1,generated_score_1,generated_distractor_2,generated_score_2,generated_distractor_3,generated_score_3
0,q_10007,the accident happened on _ .,march 1,logistic_regression,march 2,0.999118,owners,0.666688,mare,0.652047
1,q_10014,from the story we know the parrot was _ .,clever,logistic_regression,crazy,0.816743,talk man asked,0.707141,silly,0.703689
2,q_10075,what does mr brown like?,he likes music.,logistic_regression,he likes cooking.,0.992165,school popular,0.925467,englishspeaking,0.825817
3,q_10087,is xing fei a boy?,"yes,he is",logistic_regression,"yes, she does",0.993963,"no,he isn't",0.958878,eightytwo,0.922276
4,q_10130,the clean your plate campaign calls on people ...,stop wasting food,logistic_regression,wash their plates after dinner,0.992849,work part time in restaurants,0.985660,guests eaten,0.759285


,question_uid,question,correct_answer,model,generated_distractor_1,generated_score_1,generated_distractor_2,generated_score_2,generated_distractor_3,generated_score_3
0,q_10007,the accident happened on _ .,march 1,random_forest,march 2,0.999933,owners,0.867070,mare,0.421714
1,q_10014,from the story we know the parrot was _ .,clever,random_forest,kind,0.880735,thats,0.880735,crazy,0.871743
2,q_10075,what does mr brown like?,he likes music.,random_forest,he likes cooking.,0.995734,school popular,0.817861,englishspeaking,0.732178
3,q_10087,is xing fei a boy?,"yes,he is",random_forest,"yes, she does",0.992086,"no,he isn't",0.978389,eightytwo,0.937068
4,q_10130,the clean your plate campaign calls on people ...,stop wasting food,random_forest,wash their plates after dinner,0.997491,work part time in restaurants,0.995229,vacationand,0.429386


In [17]:
smooth = SmoothingFunction().method1
rouge = rouge_scorer.RougeScorer(
    ["rouge1", "rouge2", "rougeL"],
    use_stemmer=True
)

def normalize_for_eval(text):
    text = normalize_text(remove_punctuation(text))
    text = re.sub(r"\s+", " ", text).strip()
    return text

def get_reference_distractor_map(original_df):
    ref_map = {}

    for _, row in original_df.iterrows():
        refs = [
            safe_text(row["distractor_1"]),
            safe_text(row["distractor_2"]),
            safe_text(row["distractor_3"])
        ]
        refs = [r for r in refs if r.strip()]
        ref_map[row["question_uid"]] = refs

    return ref_map

def compute_text_generation_metrics(generated, reference):
    gen_tokens = tokenize_words(generated)
    ref_tokens = tokenize_words(reference)

    if not gen_tokens or not ref_tokens:
        return {
            "bleu1": 0.0,
            "bleu2": 0.0,
            "bleu4": 0.0,
            "rouge1_f1": 0.0,
            "rouge2_f1": 0.0,
            "rougeL_f1": 0.0,
            "meteor": 0.0
        }

    bleu1 = sentence_bleu(
        [ref_tokens],
        gen_tokens,
        weights=(1, 0, 0, 0),
        smoothing_function=smooth
    )

    bleu2 = sentence_bleu(
        [ref_tokens],
        gen_tokens,
        weights=(0.5, 0.5, 0, 0),
        smoothing_function=smooth
    )

    bleu4 = sentence_bleu(
        [ref_tokens],
        gen_tokens,
        weights=(0.25, 0.25, 0.25, 0.25),
        smoothing_function=smooth
    )

    rouge_scores = rouge.score(reference, generated)

    try:
        meteor = meteor_score([ref_tokens], gen_tokens)
    except:
        meteor = 0.0

    return {
        "bleu1": float(bleu1),
        "bleu2": float(bleu2),
        "bleu4": float(bleu4),
        "rouge1_f1": float(rouge_scores["rouge1"].fmeasure),
        "rouge2_f1": float(rouge_scores["rouge2"].fmeasure),
        "rougeL_f1": float(rouge_scores["rougeL"].fmeasure),
        "meteor": float(meteor)
    }

def best_match_metrics(generated, references):
    """
    Distractor order does not matter.
    Each generated distractor is compared with all 3 reference distractors.
    The best matching reference is used.
    """
    best_metrics = None
    best_score = -1

    for ref in references:
        metrics = compute_text_generation_metrics(generated, ref)
        score = (
            metrics["meteor"] +
            metrics["rougeL_f1"] +
            metrics["bleu1"]
        )

        if score > best_score:
            best_score = score
            best_metrics = metrics

    if best_metrics is None:
        return {
            "bleu1": 0.0,
            "bleu2": 0.0,
            "bleu4": 0.0,
            "rouge1_f1": 0.0,
            "rouge2_f1": 0.0,
            "rougeL_f1": 0.0,
            "meteor": 0.0
        }

    return best_metrics

def evaluate_generated_distractors_bleu_rouge_meteor(selected_df, original_df, model_name):
    ref_map = get_reference_distractor_map(original_df)

    metric_rows = []

    for _, row in selected_df.iterrows():
        qid = row["question_uid"]
        refs = ref_map.get(qid, [])

        generated_distractors = [
            safe_text(row["generated_distractor_1"]),
            safe_text(row["generated_distractor_2"]),
            safe_text(row["generated_distractor_3"])
        ]

        generated_distractors = [
            g for g in generated_distractors
            if g.strip()
        ]

        for gen in generated_distractors:
            metrics = best_match_metrics(gen, refs)
            metrics["question_uid"] = qid
            metrics["generated_distractor"] = gen
            metric_rows.append(metrics)

    metrics_df = pd.DataFrame(metric_rows)

    summary = {
        "model": model_name,
        "evaluation_type": "BLEU_ROUGE_METEOR_for_generated_distractors",
        "questions_evaluated": int(selected_df["question_uid"].nunique()),
        "generated_distractors_evaluated": int(len(metrics_df)),
        "bleu1": float(metrics_df["bleu1"].mean()),
        "bleu2": float(metrics_df["bleu2"].mean()),
        "bleu4": float(metrics_df["bleu4"].mean()),
        "rouge1_f1": float(metrics_df["rouge1_f1"].mean()),
        "rouge2_f1": float(metrics_df["rouge2_f1"].mean()),
        "rougeL_f1": float(metrics_df["rougeL_f1"].mean()),
        "meteor": float(metrics_df["meteor"].mean())
    }

    return summary, metrics_df

lr_val_generation_metrics, lr_val_generation_details = evaluate_generated_distractors_bleu_rouge_meteor(
    val_top3_lr,
    val_df,
    "logistic_regression"
)

rf_val_generation_metrics, rf_val_generation_details = evaluate_generated_distractors_bleu_rouge_meteor(
    val_top3_rf,
    val_df,
    "random_forest"
)

print("Logistic Regression validation generation metrics:")
print(json.dumps(lr_val_generation_metrics, indent=2))

print("\nRandom Forest validation generation metrics:")
print(json.dumps(rf_val_generation_metrics, indent=2))

Logistic Regression validation generation metrics:
{
  "model": "logistic_regression",
  "evaluation_type": "BLEU_ROUGE_METEOR_for_generated_distractors",
  "questions_evaluated": 5000,
  "generated_distractors_evaluated": 15000,
  "bleu1": 0.7581867303110723,
  "bleu2": 0.720795331192967,
  "bleu4": 0.6475944114800504,
  "rouge1_f1": 0.7631002364709022,
  "rouge2_f1": 0.709289074074074,
  "rougeL_f1": 0.7630258930365587,
  "meteor": 0.7276093353103879
}

Random Forest validation generation metrics:
{
  "model": "random_forest",
  "evaluation_type": "BLEU_ROUGE_METEOR_for_generated_distractors",
  "questions_evaluated": 5000,
  "generated_distractors_evaluated": 15000,
  "bleu1": 0.7993020767046078,
  "bleu2": 0.749923605437796,
  "bleu4": 0.6652072780920628,
  "rouge1_f1": 0.8034524517312752,
  "rouge2_f1": 0.7348068518518519,
  "rougeL_f1": 0.8033684786673022,
  "meteor": 0.758802411047457
}


In [18]:
distractor_generation_val_comparison = pd.DataFrame([
    lr_val_generation_metrics,
    rf_val_generation_metrics
])

distractor_generation_val_comparison_path = f"{MODEL_B_RESULTS_DIR}/model_b_distractor_generation_validation_bleu_rouge_meteor_full.csv"
distractor_generation_val_comparison.to_csv(
    distractor_generation_val_comparison_path,
    index=False
)

display(distractor_generation_val_comparison)

# Final selection uses generation metrics only.
# Priority: METEOR, ROUGE-L, BLEU-1
best_row = distractor_generation_val_comparison.sort_values(
    by=["meteor", "rougeL_f1", "bleu1"],
    ascending=False
).iloc[0]

BEST_DISTRACTOR_MODEL_NAME = best_row["model"]

if BEST_DISTRACTOR_MODEL_NAME == "logistic_regression":
    best_distractor_model = logreg_ranker
else:
    best_distractor_model = rf_ranker

print("Best distractor model based on BLEU/ROUGE/METEOR:")
print(BEST_DISTRACTOR_MODEL_NAME)
print(best_row.to_dict())
print("Saved validation generation comparison:", distractor_generation_val_comparison_path)

,model,evaluation_type,questions_evaluated,generated_distractors_evaluated,bleu1,bleu2,bleu4,rouge1_f1,rouge2_f1,rougeL_f1,meteor
0,logistic_regression,BLEU_ROUGE_METEOR_for_generated_distractors,5000,15000,0.758187,0.720795,0.647594,0.763100,0.709289,0.763026,0.727609
1,random_forest,BLEU_ROUGE_METEOR_for_generated_distractors,5000,15000,0.799302,0.749924,0.665207,0.803452,0.734807,0.803368,0.758802


Best distractor model based on BLEU/ROUGE/METEOR:
random_forest
{'model': 'random_forest', 'evaluation_type': 'BLEU_ROUGE_METEOR_for_generated_distractors', 'questions_evaluated': 5000, 'generated_distractors_evaluated': 15000, 'bleu1': 0.7993020767046078, 'bleu2': 0.749923605437796, 'bleu4': 0.6652072780920628, 'rouge1_f1': 0.8034524517312752, 'rouge2_f1': 0.7348068518518519, 'rougeL_f1': 0.8033684786673022, 'meteor': 0.758802411047457}
Saved validation generation comparison: /content/drive/MyDrive/race_rc_project/results/model_b_full/model_b_distractor_generation_validation_bleu_rouge_meteor_full.csv


In [19]:
best_test_scored_path = f"{MODEL_B_RESULTS_DIR}/test_scored_{BEST_DISTRACTOR_MODEL_NAME}_full.csv"
best_test_top3_path = f"{MODEL_B_RESULTS_DIR}/test_top3_{BEST_DISTRACTOR_MODEL_NAME}_full.csv"

test_scored_best = score_feature_file(
    best_distractor_model,
    test_features_path,
    test_candidates_path,
    best_test_scored_path,
    BEST_DISTRACTOR_MODEL_NAME,
    SCORE_CHUNK_SIZE
)

test_top3_best = select_top3_distractors(
    test_scored_best,
    best_test_top3_path,
    BEST_DISTRACTOR_MODEL_NAME,
    diversity_lambda=0.25
)

test_distractor_generation_metrics, test_distractor_generation_details = evaluate_generated_distractors_bleu_rouge_meteor(
    test_top3_best,
    test_df,
    BEST_DISTRACTOR_MODEL_NAME
)

test_distractor_generation_metrics_path = f"{MODEL_B_RESULTS_DIR}/model_b_best_distractor_test_bleu_rouge_meteor_full.json"
test_distractor_generation_details_path = f"{MODEL_B_RESULTS_DIR}/model_b_best_distractor_test_generation_details_full.csv"

with open(test_distractor_generation_metrics_path, "w") as f:
    json.dump(test_distractor_generation_metrics, f, indent=2)

test_distractor_generation_details.to_csv(
    test_distractor_generation_details_path,
    index=False
)

print("Best distractor test BLEU/ROUGE/METEOR metrics:")
print(json.dumps(test_distractor_generation_metrics, indent=2))

print("Saved summary:", test_distractor_generation_metrics_path)
print("Saved detailed scores:", test_distractor_generation_details_path)

display(test_top3_best.head(10))

[Parallel(n_jobs=2)]: Using backend ThreadingBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done  46 tasks      | elapsed:    0.3s
[Parallel(n_jobs=2)]: Done 196 tasks      | elapsed:    1.2s
[Parallel(n_jobs=2)]: Done 200 out of 200 | elapsed:    1.2s finished


random_forest: scored 100000 rows


[Parallel(n_jobs=2)]: Using backend ThreadingBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done  46 tasks      | elapsed:    0.5s
[Parallel(n_jobs=2)]: Done 196 tasks      | elapsed:    2.1s
[Parallel(n_jobs=2)]: Done 200 out of 200 | elapsed:    2.1s finished


random_forest: scored 200000 rows


[Parallel(n_jobs=2)]: Using backend ThreadingBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done  46 tasks      | elapsed:    0.1s
[Parallel(n_jobs=2)]: Done 196 tasks      | elapsed:    0.2s
[Parallel(n_jobs=2)]: Done 200 out of 200 | elapsed:    0.2s finished


random_forest: scored 214740 rows
random_forest: saved scored candidates: /content/drive/MyDrive/race_rc_project/results/model_b_full/test_scored_random_forest_full.csv
(214740, 8)
Saved selected distractors: /content/drive/MyDrive/race_rc_project/results/model_b_full/test_top3_random_forest_full.csv
(5000, 10)
Best distractor test BLEU/ROUGE/METEOR metrics:
{
  "model": "random_forest",
  "evaluation_type": "BLEU_ROUGE_METEOR_for_generated_distractors",
  "questions_evaluated": 5000,
  "generated_distractors_evaluated": 15000,
  "bleu1": 0.8020472474708141,
  "bleu2": 0.7526375868025786,
  "bleu4": 0.671321702209595,
  "rouge1_f1": 0.8060827574604046,
  "rouge2_f1": 0.7385893698893699,
  "rougeL_f1": 0.8059566679943151,
  "meteor": 0.7618069403001995
}
Saved summary: /content/drive/MyDrive/race_rc_project/results/model_b_full/model_b_best_distractor_test_bleu_rouge_meteor_full.json
Saved detailed scores: /content/drive/MyDrive/race_rc_project/results/model_b_full/model_b_best_distract

,question_uid,question,correct_answer,model,generated_distractor_1,generated_score_1,generated_distractor_2,generated_score_2,generated_distractor_3,generated_score_3
0,q_10011,the parrot was expensive because _ .,it could talk with the people,random_forest,it was pretty beautiful,0.978606,talk man asked,0.893788,it was in the middle of the store,0.689128
1,q_10019,which of the following is not right?,mr zhao's has the best food and the most comfo...,random_forest,mr cool's has the hardest seats and the worst ...,0.999741,seats comfortable,0.693653,people different,0.523013
2,q_10021,she likes to eat _ and bread in the morning.,"eggs, milk, bananas",random_forest,"eggs, chicken, milk",0.996010,salad,0.106834,pies,0.093727
3,q_10025,in england the traffic drives _ .,on the left,random_forest,in the front,0.984761,on the right,0.960778,in the middle,0.942999
4,q_10036,"which of the following should you say ""no"" whe...",stay alone at home as much as possible,random_forest,keep a diary to express your feelings,0.923203,you should always look for the good things of ...,0.917299,learn something new and go for it,0.705034
5,q_10041,"according to the passage, recycling language m...",using vocabulary that we have learnt very often,random_forest,revising vocabulary at a proper time,0.999914,learning new vocabulary as much as possible,0.999905,repeating vocabulary at times,0.997086
6,q_10070,"in the earthquake places, rats will become man...",find the positions of people alive who are tra...,random_forest,serve as food for people alive who are trapped...,0.999983,take the place of man in rescue jobs,0.998955,send signals for the coming earthquake,0.998670
7,q_10071,"in doing rescue jobs, _ .",robots' sense of smell can be affected by othe...,random_forest,dogs don't need to be trained to smell people.,0.999860,rats can see in the dark and are smaller than ...,0.999786,rats have better sense of smell than dogs.,0.999625
8,q_10089,_ to take class notes is more helpful for a lo...,writing by hand,random_forest,recording with a computer,0.988906,typing on a keyboard,0.980080,took notes writing,0.455151
9,q_10094,the passage is mainly about _ .,the changing global diet,random_forest,the increasing types of diet,0.998215,the reason why people dislike fast food,0.977533,fastfood places,0.912752


In [20]:
def mask_answer_in_sentence(sentence, answer):
    sentence = safe_text(sentence)
    answer = safe_text(answer)

    if not answer.strip():
        return sentence

    pattern = re.compile(re.escape(answer), re.IGNORECASE)
    masked = pattern.sub("[ANSWER]", sentence)

    # If exact full answer not found, mask answer content words.
    if masked == sentence:
        for w in content_words(answer):
            masked = re.sub(rf"\b{re.escape(w)}\b", "[ANSWER]", masked, flags=re.IGNORECASE)

    return masked

def sentence_feature_rows_for_question(row):
    qid = row["question_uid"]
    question = safe_text(row["question"])
    answer = safe_text(row["correct_answer"])
    sentences = parse_sentence_list(row["article_sentences"])

    rows = []

    if not sentences:
        return rows

    # Proxy label: sentence most related to question + answer.
    combined_query = question + " " + answer

    sent_vec = model_b_tfidf.transform(sentences)
    q_vec = model_b_tfidf.transform([question] * len(sentences))
    a_vec = model_b_tfidf.transform([answer] * len(sentences))
    qa_vec = model_b_tfidf.transform([combined_query] * len(sentences))

    cos_q = paired_cosine_sparse(sent_vec, q_vec)
    cos_a = paired_cosine_sparse(sent_vec, a_vec)
    cos_qa = paired_cosine_sparse(sent_vec, qa_vec)

    proxy_scores = []

    for i, sentence in enumerate(sentences):
        s = safe_text(sentence)
        score = (
            0.45 * cos_q[i] +
            0.35 * cos_a[i] +
            0.10 * word_overlap(s, question) +
            0.10 * word_overlap(s, answer)
        )
        proxy_scores.append(score)

    best_idx = int(np.argmax(proxy_scores))

    for i, sentence in enumerate(sentences):
        s = safe_text(sentence)
        sent_words = content_words(s)

        rows.append({
            "question_uid": qid,
            "sentence_index": i,
            "sentence": s,
            "question": question,
            "correct_answer": answer,
            "cos_question_sentence": float(cos_q[i]),
            "cos_answer_sentence": float(cos_a[i]),
            "cos_question_answer_sentence": float(cos_qa[i]),
            "question_sentence_overlap": word_overlap(question, s),
            "answer_sentence_overlap": word_overlap(answer, s),
            "sentence_position_norm": i / max(1, len(sentences) - 1),
            "sentence_word_len": len(sent_words),
            "contains_correct_answer": int(normalize_text(answer) in normalize_text(s)),
            "proxy_relevance_score": float(proxy_scores[i]),
            "label": int(i == best_idx)
        })

    return rows

print("Hint feature functions ready.")

Hint feature functions ready.


In [21]:
def build_hint_feature_dataset(df, name, output_path):
    if os.path.exists(output_path):
        print(f"{name}: hint feature file already exists. Loading:", output_path)
        hint_df = pd.read_csv(output_path)
        print(hint_df.shape)
        print(hint_df["label"].value_counts())
        return hint_df

    all_rows = []

    for i, row in df.iterrows():
        all_rows.extend(sentence_feature_rows_for_question(row))

        if (i + 1) % 5000 == 0:
            print(f"{name}: processed {i + 1}/{len(df)} questions")

    hint_df = pd.DataFrame(all_rows)
    hint_df.to_csv(output_path, index=False)

    print(f"{name} hint feature dataset saved:", output_path)
    print(hint_df.shape)
    print(hint_df["label"].value_counts())

    return hint_df

train_hint_features_path = f"{MODEL_B_RESULTS_DIR}/model_b_train_hint_features_full.csv"
val_hint_features_path = f"{MODEL_B_RESULTS_DIR}/model_b_val_hint_features_full.csv"
test_hint_features_path = f"{MODEL_B_RESULTS_DIR}/model_b_test_hint_features_full.csv"

train_hint_features = build_hint_feature_dataset(
    train_df,
    "train_hint",
    train_hint_features_path
)

val_hint_features = build_hint_feature_dataset(
    val_df,
    "val_hint",
    val_hint_features_path
)

test_hint_features = build_hint_feature_dataset(
    test_df,
    "test_hint",
    test_hint_features_path
)

train_hint: processed 5000/70280 questions
train_hint: processed 10000/70280 questions
train_hint: processed 15000/70280 questions
train_hint: processed 20000/70280 questions
train_hint: processed 25000/70280 questions
train_hint: processed 30000/70280 questions
train_hint: processed 35000/70280 questions
train_hint: processed 40000/70280 questions
train_hint: processed 45000/70280 questions
train_hint: processed 50000/70280 questions
train_hint: processed 55000/70280 questions
train_hint: processed 60000/70280 questions
train_hint: processed 65000/70280 questions
train_hint: processed 70000/70280 questions
train_hint hint feature dataset saved: /content/drive/MyDrive/race_rc_project/results/model_b_full/model_b_train_hint_features_full.csv
(1186529, 15)
label
0    1116249
1      70280
Name: count, dtype: int64
val_hint: processed 5000/8785 questions
val_hint hint feature dataset saved: /content/drive/MyDrive/race_rc_project/results/model_b_full/model_b_val_hint_features_full.csv
(1496

In [22]:
hint_feature_cols = [
    "cos_question_sentence",
    "cos_answer_sentence",
    "cos_question_answer_sentence",
    "question_sentence_overlap",
    "answer_sentence_overlap",
    "sentence_position_norm",
    "sentence_word_len",
    "contains_correct_answer"
]

X_hint_train = train_hint_features[hint_feature_cols]
y_hint_train = train_hint_features["label"].astype(int)

X_hint_val = val_hint_features[hint_feature_cols]
y_hint_val = val_hint_features["label"].astype(int)

hint_ranker_path = f"{MODEL_B_DIR}/logistic_regression_hint_ranker_full.pkl"

if os.path.exists(hint_ranker_path):
    hint_ranker = joblib.load(hint_ranker_path)
    print("Loaded hint ranker:", hint_ranker_path)
else:
    hint_ranker = Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=RANDOM_STATE,
            n_jobs=-1
        ))
    ])

    hint_ranker.fit(X_hint_train, y_hint_train)
    joblib.dump(hint_ranker, hint_ranker_path)
    print("Saved hint ranker:", hint_ranker_path)

hint_val_pred = hint_ranker.predict(X_hint_val)

hint_val_metrics = {
    "model": "Logistic Regression Hint Ranker",
    "accuracy": accuracy_score(y_hint_val, hint_val_pred),
    "precision": precision_score(y_hint_val, hint_val_pred, zero_division=0),
    "recall": recall_score(y_hint_val, hint_val_pred, zero_division=0),
    "f1": f1_score(y_hint_val, hint_val_pred, zero_division=0)
}

print(json.dumps(hint_val_metrics, indent=2))
print(confusion_matrix(y_hint_val, hint_val_pred))
print(classification_report(y_hint_val, hint_val_pred, zero_division=0))

Saved hint ranker: /content/drive/MyDrive/race_rc_project/models/model_b/logistic_regression_hint_ranker_full.pkl
{
  "model": "Logistic Regression Hint Ranker",
  "accuracy": 0.8680631211693859,
  "precision": 0.2870748299319728,
  "recall": 0.8406374501992032,
  "f1": 0.4279918864097363
}
[[122492  18340]
 [  1400   7385]]
              precision    recall  f1-score   support

           0       0.99      0.87      0.93    140832
           1       0.29      0.84      0.43      8785

    accuracy                           0.87    149617
   macro avg       0.64      0.86      0.68    149617
weighted avg       0.95      0.87      0.90    149617



In [23]:
def score_hint_features(hint_df, model):
    hint_df = hint_df.copy()
    X = hint_df[hint_feature_cols]

    if hasattr(model, "predict_proba"):
        hint_df["hint_score"] = model.predict_proba(X)[:, 1]
    else:
        hint_df["hint_score"] = model.decision_function(X)

    return hint_df

def generate_graduated_hints_for_question(row, hint_ranker):
    feature_rows = sentence_feature_rows_for_question(row)

    if not feature_rows:
        return {
            "hint_1": "Think about the main idea of the passage.",
            "hint_2": "Look for the sentence most related to the question.",
            "hint_3": "Review the passage carefully near the relevant detail."
        }

    hdf = pd.DataFrame(feature_rows)
    hdf = score_hint_features(hdf, hint_ranker)
    hdf = hdf.sort_values("hint_score", ascending=False)

    best_sentence = safe_text(hdf.iloc[0]["sentence"])
    answer = safe_text(row["correct_answer"])
    question = safe_text(row["question"])

    q_keywords = content_words(question)
    keyword_text = ", ".join(q_keywords[:5]) if q_keywords else "the important words in the question"

    masked_sentence = mask_answer_in_sentence(best_sentence, answer)

    # Strong hint should guide but avoid directly revealing answer.
    strong_hint = masked_sentence
    if normalize_text(answer) in normalize_text(strong_hint):
        strong_hint = mask_answer_in_sentence(strong_hint, answer)

    return {
        "hint_1": f"Focus on the part of the passage related to: {keyword_text}.",
        "hint_2": f"A relevant sentence says: {masked_sentence}",
        "hint_3": f"The answer is supported by this context: {strong_hint}"
    }

sample_hint = generate_graduated_hints_for_question(test_df.iloc[0], hint_ranker)

print("Question:", test_df.iloc[0]["question"])
print("Correct answer:", test_df.iloc[0]["correct_answer"])
print(json.dumps(sample_hint, indent=2))

Question: according to the passage, some people think that if you don't have breakfast you will _ .
Correct answer: lose weight
{
  "hint_1": "Focus on the part of the passage related to: according, passage, people, think, dont.",
  "hint_2": "A relevant sentence says: you will [ANSWER] more [ANSWER] if you your other meals.",
  "hint_3": "The answer is supported by this context: you will [ANSWER] more [ANSWER] if you your other meals."
}


In [24]:
def get_proxy_hint_reference(row):
    """
    RACE has no gold hints.
    Proxy reference = most question/answer-relevant sentence with answer masked.
    """
    feature_rows = sentence_feature_rows_for_question(row)

    if not feature_rows:
        return "Look for the sentence most related to the question."

    hdf = pd.DataFrame(feature_rows)
    hdf = hdf.sort_values("proxy_relevance_score", ascending=False)

    best_sentence = safe_text(hdf.iloc[0]["sentence"])
    answer = safe_text(row["correct_answer"])

    return mask_answer_in_sentence(best_sentence, answer)

def evaluate_generated_hints_bleu_rouge_meteor(df, hint_ranker, max_rows=None):
    rows = []

    eval_df = df.copy()

    if max_rows is not None:
        eval_df = eval_df.head(max_rows)

    for i, (_, row) in enumerate(eval_df.iterrows()):
        generated_hints = generate_graduated_hints_for_question(row, hint_ranker)
        reference_hint = get_proxy_hint_reference(row)

        # Evaluate the more content-rich hints.
        generated_hint_text = (
            generated_hints["hint_2"] + " " +
            generated_hints["hint_3"]
        )

        metrics = compute_text_generation_metrics(
            generated_hint_text,
            reference_hint
        )

        metrics["question_uid"] = row["question_uid"]
        metrics["question"] = safe_text(row["question"])
        metrics["correct_answer"] = safe_text(row["correct_answer"])
        metrics["reference_hint"] = reference_hint
        metrics["generated_hint_1"] = generated_hints["hint_1"]
        metrics["generated_hint_2"] = generated_hints["hint_2"]
        metrics["generated_hint_3"] = generated_hints["hint_3"]

        rows.append(metrics)

        if (i + 1) % 1000 == 0:
            print(f"Evaluated hints for {i + 1}/{len(eval_df)} questions")

    details_df = pd.DataFrame(rows)

    summary = {
        "model": "Logistic Regression Hint Ranker",
        "evaluation_type": "BLEU_ROUGE_METEOR_for_generated_hints",
        "important_limitation": "RACE has no gold human-written hints, so the reference hint is a proxy: the most relevant passage sentence with the correct answer masked.",
        "questions_evaluated": int(len(details_df)),
        "bleu1": float(details_df["bleu1"].mean()),
        "bleu2": float(details_df["bleu2"].mean()),
        "bleu4": float(details_df["bleu4"].mean()),
        "rouge1_f1": float(details_df["rouge1_f1"].mean()),
        "rouge2_f1": float(details_df["rouge2_f1"].mean()),
        "rougeL_f1": float(details_df["rougeL_f1"].mean()),
        "meteor": float(details_df["meteor"].mean())
    }

    return summary, details_df

hint_generation_test_metrics, hint_generation_test_details = evaluate_generated_hints_bleu_rouge_meteor(
    test_df,
    hint_ranker,
    max_rows=None
)

hint_generation_test_metrics_path = f"{MODEL_B_RESULTS_DIR}/model_b_hint_generation_test_bleu_rouge_meteor_full.json"
hint_generation_test_details_path = f"{MODEL_B_RESULTS_DIR}/model_b_hint_generation_test_details_full.csv"

with open(hint_generation_test_metrics_path, "w") as f:
    json.dump(hint_generation_test_metrics, f, indent=2)

hint_generation_test_details.to_csv(
    hint_generation_test_details_path,
    index=False
)

print("Hint generation BLEU/ROUGE/METEOR metrics:")
print(json.dumps(hint_generation_test_metrics, indent=2))

print("Saved hint summary:", hint_generation_test_metrics_path)
print("Saved hint details:", hint_generation_test_details_path)

display(hint_generation_test_details.head(10)[[
    "question",
    "correct_answer",
    "reference_hint",
    "generated_hint_1",
    "generated_hint_2",
    "generated_hint_3",
    "bleu1",
    "rougeL_f1",
    "meteor"
]])

Evaluated hints for 1000/8786 questions
Evaluated hints for 2000/8786 questions
Evaluated hints for 3000/8786 questions
Evaluated hints for 4000/8786 questions
Evaluated hints for 5000/8786 questions
Evaluated hints for 6000/8786 questions
Evaluated hints for 7000/8786 questions
Evaluated hints for 8000/8786 questions
Hint generation BLEU/ROUGE/METEOR metrics:
{
  "model": "Logistic Regression Hint Ranker",
  "evaluation_type": "BLEU_ROUGE_METEOR_for_generated_hints",
  "important_limitation": "RACE has no gold human-written hints, so the reference hint is a proxy: the most relevant passage sentence with the correct answer masked.",
  "questions_evaluated": 8786,
  "bleu1": 0.29553114699769656,
  "bleu2": 0.27734729376701284,
  "bleu4": 0.25695117753951857,
  "rouge1_f1": 0.43800344820527587,
  "rouge2_f1": 0.39865122367358485,
  "rougeL_f1": 0.4337181534059404,
  "meteor": 0.6869130950039368
}
Saved hint summary: /content/drive/MyDrive/race_rc_project/results/model_b_full/model_b_hint

,question,correct_answer,reference_hint,generated_hint_1,generated_hint_2,generated_hint_3,bleu1,rougeL_f1,meteor
0,"according to the passage, some people think th...",lose weight,you will [ANSWER] more [ANSWER] if you your ot...,Focus on the part of the passage related to: a...,A relevant sentence says: you will [ANSWER] mo...,The answer is supported by this context: you w...,0.322581,0.487805,0.826033
1,the writer was surprised to find the woman's w...,clean,her window was [ANSWER]!,Focus on the part of the passage related to: w...,A relevant sentence says: her window was [ANSW...,The answer is supported by this context: her w...,0.210526,0.347826,0.721591
2,"according to the passage, the fa(football asso...",each may.,the most interesting part of the english footb...,Focus on the part of the passage related to: a...,A relevant sentence says: the most interesting...,The answer is supported by this context: the m...,0.372093,0.548387,0.855511
3,attention exchange was not a major concern in ...,natural checks and balances,"in traditional societies, with smaller populat...",Focus on the part of the passage related to: a...,A relevant sentence says: there were [ANSWER]:...,The answer is supported by this context: there...,0.054054,0.078431,0.092025
4,"according to the passage, a happy person is mo...",a kind helper,"only when you are at peace with yourself, will...",Focus on the part of the passage related to: a...,A relevant sentence says: only when you are at...,The answer is supported by this context: only ...,0.403509,0.575000,0.871176
5,"according to the article, london _ .",exists on a river,"thus, greater london is made up of the city an...",Focus on the part of the passage related to: a...,A relevant sentence says: it is surrounded by ...,The answer is supported by this context: it is...,0.082353,0.112150,0.190217
6,why didn't the author give any advice to lydia...,because she was too tired to talk to lydia.,"it was my friend [ANSWER],upset over an argume...",Focus on the part of the passage related to: d...,A relevant sentence says: it was my friend [AN...,The answer is supported by this context: it wa...,0.333333,0.510638,0.833020
7,which of the following is probably a better ti...,4 pm.,"12am--next up is the tanah lot temple, perhaps...",Focus on the part of the passage related to: f...,A relevant sentence says: 12am--next up is the...,The answer is supported by this context: 12am-...,0.351351,0.535714,0.843964
8,what did the students find out about mckay?,he trained pilots for some time.,"eddie mckay, a once-forgotten pilot, is a subj...",Focus on the part of the passage related to: d...,"A relevant sentence says: eddie mckay, a once-...",The answer is supported by this context: eddie...,0.387755,0.563380,0.863573
9,my spanish friends wanted advice about _ .,finding places to stay in england,"when i lived in spain,some spanish friends of ...",Focus on the part of the passage related to: s...,"A relevant sentence says: ""we didn't [ANSWER] ...","The answer is supported by this context: ""we d...",0.124424,0.147910,0.243655


In [25]:
def generate_model_b_output_for_row(row, distractor_model, hint_ranker, max_candidates=40):
    # Build inference candidates from passage only.
    candidates = extract_candidate_phrases(
        article=row["article"],
        question=row["question"],
        correct_answer=row["correct_answer"],
        max_candidates=max_candidates
    )

    temp_rows = []
    for cand in candidates:
        temp_rows.append({
            "question_uid": row["question_uid"],
            "article": row["article"],
            "question": row["question"],
            "correct_answer": row["correct_answer"],
            "candidate": cand,
            "label": 0,
            "source": "inference_extracted"
        })

    if not temp_rows:
        distractors = ["", "", ""]
    else:
        temp_candidate_df = pd.DataFrame(temp_rows)
        temp_features = extract_features_chunk(temp_candidate_df)
        X_temp = temp_features[feature_cols]

        temp_candidate_df["score"] = distractor_model.predict_proba(X_temp)[:, 1]

        selected = select_top3_for_group(temp_candidate_df, diversity_lambda=0.25)
        distractors = [s["candidate"] for s in selected]

        while len(distractors) < 3:
            distractors.append("")

    hints = generate_graduated_hints_for_question(row, hint_ranker)

    return {
        "question_uid": row["question_uid"],
        "article": row["article"],
        "question": row["question"],
        "correct_answer": row["correct_answer"],
        "generated_distractor_1": distractors[0],
        "generated_distractor_2": distractors[1],
        "generated_distractor_3": distractors[2],
        "hint_1": hints["hint_1"],
        "hint_2": hints["hint_2"],
        "hint_3": hints["hint_3"]
    }

examples = []

for i in range(10):
    examples.append(generate_model_b_output_for_row(
        test_df.iloc[i],
        best_distractor_model,
        hint_ranker,
        max_candidates=40
    ))

examples_df = pd.DataFrame(examples)

examples_path = f"{MODEL_B_RESULTS_DIR}/model_b_qualitative_examples_full.csv"
examples_df.to_csv(examples_path, index=False)

display(examples_df[[
    "question",
    "correct_answer",
    "generated_distractor_1",
    "generated_distractor_2",
    "generated_distractor_3",
    "hint_1",
    "hint_2",
    "hint_3"
]])

print("Saved qualitative examples:", examples_path)

[Parallel(n_jobs=2)]: Using backend ThreadingBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done  46 tasks      | elapsed:    0.0s
[Parallel(n_jobs=2)]: Done 196 tasks      | elapsed:    0.0s
[Parallel(n_jobs=2)]: Done 200 out of 200 | elapsed:    0.0s finished
[Parallel(n_jobs=2)]: Using backend ThreadingBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done  46 tasks      | elapsed:    0.0s
[Parallel(n_jobs=2)]: Done 196 tasks      | elapsed:    0.1s
[Parallel(n_jobs=2)]: Done 200 out of 200 | elapsed:    0.1s finished
[Parallel(n_jobs=2)]: Using backend ThreadingBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done  46 tasks      | elapsed:    0.0s
[Parallel(n_jobs=2)]: Done 196 tasks      | elapsed:    0.1s
[Parallel(n_jobs=2)]: Done 200 out of 200 | elapsed:    0.1s finished
[Parallel(n_jobs=2)]: Using backend ThreadingBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done  46 tasks      | elapsed:    0.0s
[Parallel(n_jobs=2)]: Done 196 tasks      |

,question,correct_answer,generated_distractor_1,generated_distractor_2,generated_distractor_3,hint_1,hint_2,hint_3
0,"according to the passage, some people think th...",lose weight,kinds breakfasts,dont,egg,Focus on the part of the passage related to: a...,A relevant sentence says: you will [ANSWER] mo...,The answer is supported by this context: you w...
1,the writer was surprised to find the woman's w...,clean,didnt,clearly,dirty,Focus on the part of the passage related to: w...,A relevant sentence says: her window was [ANSW...,The answer is supported by this context: her w...
2,"according to the passage, the fa(football asso...",each may.,years,worlds,charlton,Focus on the part of the passage related to: a...,A relevant sentence says: the most interesting...,The answer is supported by this context: the m...
3,attention exchange was not a major concern in ...,natural checks and balances,attention people,loved ones,materialistic,Focus on the part of the passage related to: a...,A relevant sentence says: there were [ANSWER]:...,The answer is supported by this context: there...
4,"according to the passage, a happy person is mo...",a kind helper,surebut,dont,youre,Focus on the part of the passage related to: a...,A relevant sentence says: only when you are at...,The answer is supported by this context: only ...
5,"according to the article, london _ .",exists on a river,ring boroughs called,administrative,nearly thirty,Focus on the part of the passage related to: a...,A relevant sentence says: it is surrounded by ...,The answer is supported by this context: it is...
6,why didn't the author give any advice to lydia...,because she was too tired to talk to lydia.,lydiaupset,timeexhausted,advicebut,Focus on the part of the passage related to: d...,A relevant sentence says: it was my friend [AN...,The answer is supported by this context: it wa...
7,which of the following is probably a better ti...,4 pm.,indonesias,heres,tip,Focus on the part of the passage related to: f...,A relevant sentence says: 12am--next up is the...,The answer is supported by this context: 12am-...
8,what did the students find out about mckay?,he trained pilots for some time.,onceforgotten,mckays,exhibiting case,Focus on the part of the passage related to: d...,"A relevant sentence says: eddie mckay, a once-...",The answer is supported by this context: eddie...
9,my spanish friends wanted advice about _ .,finding places to stay in england,stay bed breakfast,houses sign,spainsome,Focus on the part of the passage related to: s...,"A relevant sentence says: ""we didn't [ANSWER] ...","The answer is supported by this context: ""we d..."


Saved qualitative examples: /content/drive/MyDrive/race_rc_project/results/model_b_full/model_b_qualitative_examples_full.csv


In [26]:
model_b_summary = {
    "model_b_task": "Distractor and Hint Generator",
    "updated_evaluation_policy": "Final reporting uses BLEU, ROUGE, and METEOR only, following the latest project announcement.",

    "data_files": {
        "train": TRAIN_PATH,
        "val": VAL_PATH,
        "test": TEST_PATH
    },

    "full_mode": FULL_MODE,

    "distractor_generation": {
        "candidate_extraction": "Unigrams, bigrams, trigrams, and frequency-ranked content phrases extracted from the passage.",
        "positive_training_candidates": "Original RACE wrong options: distractor_1, distractor_2, distractor_3.",
        "negative_training_candidates": "Extracted non-answer passage candidates.",
        "features": feature_cols,
        "models_trained": [
            "Logistic Regression",
            "Random Forest"
        ],
        "validation_generation_metrics": {
            "logistic_regression": lr_val_generation_metrics,
            "random_forest": rf_val_generation_metrics
        },
        "selected_model": BEST_DISTRACTOR_MODEL_NAME,
        "selection_criteria": "Highest validation METEOR, then ROUGE-L F1, then BLEU-1.",
        "test_generation_metrics": test_distractor_generation_metrics,
        "diversity_penalty": 0.25
    },

    "hint_generation": {
        "method": "ML-scored extractive sentence ranking with graduated hint templates.",
        "features": hint_feature_cols,
        "model": "Logistic Regression",
        "evaluation_limitation": "RACE does not provide gold human-written hints. Proxy reference hints are created from the most relevant passage sentence with the answer masked.",
        "test_generation_metrics": hint_generation_test_metrics
    },

    "artifacts": {
        "tfidf_vectorizer": tfidf_path,
        "logistic_regression_distractor_ranker": logreg_ranker_path,
        "random_forest_distractor_ranker": rf_ranker_path,
        "hint_ranker": hint_ranker_path,
        "test_top3_outputs": best_test_top3_path,
        "distractor_test_metrics": test_distractor_generation_metrics_path,
        "hint_test_metrics": hint_generation_test_metrics_path,
        "qualitative_examples": examples_path
    }
}

summary_path = f"{MODEL_B_RESULTS_DIR}/model_b_final_summary_full.json"

with open(summary_path, "w") as f:
    json.dump(model_b_summary, f, indent=2)

print("Saved Model B final summary:", summary_path)
print(json.dumps(model_b_summary, indent=2))

Saved Model B final summary: /content/drive/MyDrive/race_rc_project/results/model_b_full/model_b_final_summary_full.json
{
  "model_b_task": "Distractor and Hint Generator",
  "updated_evaluation_policy": "Final reporting uses BLEU, ROUGE, and METEOR only, following the latest project announcement.",
  "data_files": {
    "train": "/content/drive/MyDrive/race_rc_project/data/processed/model_b_train_full.csv",
    "val": "/content/drive/MyDrive/race_rc_project/data/processed/model_b_val_full.csv",
    "test": "/content/drive/MyDrive/race_rc_project/data/processed/model_b_test_full.csv"
  },
  "full_mode": true,
  "distractor_generation": {
    "candidate_extraction": "Unigrams, bigrams, trigrams, and frequency-ranked content phrases extracted from the passage.",
    "positive_training_candidates": "Original RACE wrong options: distractor_1, distractor_2, distractor_3.",
    "negative_training_candidates": "Extracted non-answer passage candidates.",
    "features": [
      "cos_candidate

In [31]:
final_files = [
    tfidf_path,
    logreg_ranker_path,
    rf_ranker_path,
    hint_ranker_path,
    train_candidates_path,
    val_candidates_path,
    test_candidates_path,
    train_features_path,
    val_features_path,
    test_features_path,
    best_test_top3_path,
    examples_path,
    summary_path
]

print("Final Model B files:")
for path in final_files:
    print(os.path.exists(path), "->", path)

Final Model B files:
True -> /content/drive/MyDrive/race_rc_project/models/model_b/model_b_tfidf_vectorizer_full.pkl
True -> /content/drive/MyDrive/race_rc_project/models/model_b/logistic_regression_distractor_ranker_full.pkl
True -> /content/drive/MyDrive/race_rc_project/models/model_b/random_forest_distractor_ranker_full.pkl
True -> /content/drive/MyDrive/race_rc_project/models/model_b/logistic_regression_hint_ranker_full.pkl
True -> /content/drive/MyDrive/race_rc_project/results/model_b_full/model_b_train_candidates_full.csv
True -> /content/drive/MyDrive/race_rc_project/results/model_b_full/model_b_val_candidates_full.csv
True -> /content/drive/MyDrive/race_rc_project/results/model_b_full/model_b_test_candidates_full.csv
True -> /content/drive/MyDrive/race_rc_project/results/model_b_full/model_b_train_features_full.csv
True -> /content/drive/MyDrive/race_rc_project/results/model_b_full/model_b_val_features_full.csv
True -> /content/drive/MyDrive/race_rc_project/results/model_b_full